In [2]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================
HRV_1MIN_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/Wearable_v2_HRV_1min_discard.csv"
HRV_2MIN_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/Wearable_v2_HRV_2min_discard.csv"

OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/cls_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.30

LABEL_COL = "status"
SUBJECT_COL = "subject"

HRV_FEATURE_COLS = [
    "n_peaks",
    "ibi_pass_ratio",
    "valid_ibi_count",
    "ibi_mean_sec_raw",
    "ibi_mean_sec",
    "ibi_std_sec",
    "RMSSD",
    "SDNN",
    "LF",
    "HF",
    "LF_HF",
]

CLASSIFIER_ORDER = ["DT", "LR", "RF", "SVC"]


# =============================================================================
# Classifiers
# =============================================================================
def get_classifiers():
    return {
        "DT": DecisionTreeClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced"
        ),
        "LR": LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE,
            class_weight="balanced"
        ),
        "RF": RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced"
        ),
        "SVC": SVC(
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE,
            class_weight="balanced"
        ),
    }


# =============================================================================
# Data
# =============================================================================
def read_csv_auto(path):
    sep = "\t" if open(path, "r", encoding="utf-8-sig").readline().count("\t") > 3 else ","
    df = pd.read_csv(path, sep=sep)
    df.columns = df.columns.str.strip()
    return df


def prepare_hrv_df(path, dataset_name):
    df = read_csv_auto(path)

    df = df[df[LABEL_COL].isin([0, 1])].copy()
    df["label_major"] = df[LABEL_COL].astype(int)
    df["dataset"] = dataset_name

    feature_cols = [c for c in HRV_FEATURE_COLS if c in df.columns]

    for c in feature_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    before = len(df)
    df = df.dropna(subset=feature_cols + ["label_major", SUBJECT_COL]).copy()
    after = len(df)

    print(f"\n[{dataset_name}]")
    print(f"Loaded       : {before}")
    print(f"After dropna : {after}")
    print(f"Class dist   : {df['label_major'].value_counts().to_dict()}")
    print(f"Subjects     : {sorted(df[SUBJECT_COL].unique())}")
    print(f"Features     : {feature_cols}")

    return df, feature_cols


# =============================================================================
# Evaluation
# =============================================================================
def evaluate_one_model(clf, X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

    try:
        y_prob = clf.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) == 2 else np.nan
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    return {
        "Accuracy": acc,
        "F1": f1,
        "AUC": auc,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1],
    }


def run_personalized(df, feature_cols, dataset_name):
    results = []

    for subj in sorted(df[SUBJECT_COL].unique()):
        df_subj = df[df[SUBJECT_COL] == subj].copy()
        df_subj = df_subj.dropna(subset=feature_cols + ["label_major"])

        X = df_subj[feature_cols].values.astype(float)
        y = df_subj["label_major"].values.astype(int)

        if len(df_subj) < 8 or len(np.unique(y)) < 2:
            print(f"[SKIP Personalized] {dataset_name} {subj}: insufficient samples/classes")
            continue

        class_counts = dict(zip(*np.unique(y, return_counts=True)))
        if min(class_counts.values()) < 2:
            print(f"[SKIP Personalized] {dataset_name} {subj}: too few minority samples {class_counts}")
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y,
        )

        for clf_name, clf in get_classifiers().items():
            metrics = evaluate_one_model(clf, X_train, X_test, y_train, y_test)

            row = {
                "Feature Set": dataset_name,
                "Evaluation": "Personalized",
                "Subject": subj,
                "Classifier": clf_name,
                "N": len(df_subj),
                "N_train": len(y_train),
                "N_test": len(y_test),
                "N_normal": class_counts.get(0, 0),
                "N_stress": class_counts.get(1, 0),
            }
            row.update(metrics)
            results.append(row)

    return pd.DataFrame(results)


def run_loso(df, feature_cols, dataset_name):
    results = []

    subjects = sorted(df[SUBJECT_COL].unique())

    for test_subj in subjects:
        df_train = df[df[SUBJECT_COL] != test_subj].copy()
        df_test = df[df[SUBJECT_COL] == test_subj].copy()

        df_train = df_train.dropna(subset=feature_cols + ["label_major"])
        df_test = df_test.dropna(subset=feature_cols + ["label_major"])

        X_train = df_train[feature_cols].values.astype(float)
        y_train = df_train["label_major"].values.astype(int)

        X_test = df_test[feature_cols].values.astype(float)
        y_test = df_test["label_major"].values.astype(int)

        if len(df_test) < 2 or len(np.unique(y_train)) < 2:
            print(f"[SKIP LOSO] {dataset_name} {test_subj}: insufficient data")
            continue

        test_dist = dict(zip(*np.unique(y_test, return_counts=True)))

        for clf_name, clf in get_classifiers().items():
            metrics = evaluate_one_model(clf, X_train, X_test, y_train, y_test)

            row = {
                "Feature Set": dataset_name,
                "Evaluation": "LOSO",
                "Subject": test_subj,
                "Classifier": clf_name,
                "N_train": len(y_train),
                "N_test": len(y_test),
                "N_test_normal": test_dist.get(0, 0),
                "N_test_stress": test_dist.get(1, 0),
            }
            row.update(metrics)
            results.append(row)

    return pd.DataFrame(results)


# =============================================================================
# Summary table like requested image
# =============================================================================
def format_mean_std(mean, std):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        std = 0.0
    return f"{mean:.4f}±{std:.4f}"


def make_summary_table(df_res, evaluation_name):
    rows = []

    feature_order = ["HRV 1-min", "HRV 2-min"]

    for feature_set in feature_order:
        for clf_name in CLASSIFIER_ORDER:
            sub = df_res[
                (df_res["Evaluation"] == evaluation_name)
                & (df_res["Feature Set"] == feature_set)
                & (df_res["Classifier"] == clf_name)
            ]

            if sub.empty:
                rows.append({
                    "Feature Set": feature_set,
                    "Classifier": clf_name,
                    "Accuracy": "",
                    "F1": "",
                    "AUC": "",
                })
                continue

            rows.append({
                "Feature Set": feature_set,
                "Classifier": clf_name,
                "Accuracy": format_mean_std(sub["Accuracy"].mean(), sub["Accuracy"].std()),
                "F1": format_mean_std(sub["F1"].mean(), sub["F1"].std()),
                "AUC": format_mean_std(sub["AUC"].mean(), sub["AUC"].std()),
            })

    return pd.DataFrame(rows)


def save_summary_outputs(df_res):
    for eval_name in ["Personalized", "LOSO"]:
        summary_df = make_summary_table(df_res, eval_name)

        csv_path = os.path.join(OUTPUT_DIR, f"HRV_{eval_name}_summary_table.csv")
        xlsx_path = os.path.join(OUTPUT_DIR, f"HRV_{eval_name}_summary_table.xlsx")

        summary_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            summary_df.to_excel(writer, index=False, sheet_name=eval_name)

            ws = writer.book[eval_name]
            ws.column_dimensions["A"].width = 18
            ws.column_dimensions["B"].width = 14
            ws.column_dimensions["C"].width = 18
            ws.column_dimensions["D"].width = 18
            ws.column_dimensions["E"].width = 18

            for row in ws.iter_rows():
                for cell in row:
                    cell.alignment = cell.alignment.copy(horizontal="center", vertical="center")

        print(f"[Saved] {csv_path}")
        print(f"[Saved] {xlsx_path}")
        print(f"\n[{eval_name} summary]")
        print(summary_df.to_string(index=False))


# =============================================================================
# Main
# =============================================================================
if __name__ == "__main__":
    configs = [
        ("HRV 1-min", HRV_1MIN_PATH),
        ("HRV 2-min", HRV_2MIN_PATH),
    ]

    all_results = []

    for dataset_name, path in configs:
        df, feature_cols = prepare_hrv_df(path, dataset_name)

        personalized_res = run_personalized(df, feature_cols, dataset_name)
        loso_res = run_loso(df, feature_cols, dataset_name)

        all_results.append(personalized_res)
        all_results.append(loso_res)

    df_all_results = pd.concat(all_results, axis=0, ignore_index=True)

    save_summary_outputs(df_all_results)

    print("\nDone.")


[HRV 1-min]
Loaded       : 1085
After dropna : 1083
Class dist   : {0: 882, 1: 201}
Subjects     : ['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13', 'f15', 'f16', 'f17', 'f18']
Features     : ['n_peaks', 'ibi_pass_ratio', 'valid_ibi_count', 'ibi_mean_sec_raw', 'ibi_mean_sec', 'ibi_std_sec', 'RMSSD', 'SDNN', 'LF', 'HF', 'LF_HF']

[HRV 2-min]
Loaded       : 499
After dropna : 499
Class dist   : {0: 414, 1: 85}
Subjects     : ['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13', 'f15', 'f16', 'f17', 'f18']
Features     : ['n_peaks', 'ibi_pass_ratio', 'valid_ibi_count', 'ibi_mean_sec_raw', 'ibi_mean_sec', 'ibi_std_sec', 'RMSSD', 'SDNN', 'LF', 'HF', 'LF_HF']
[Saved] /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/cls_results/HRV_Personalized_summary_table.csv
[Saved] /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/cls_results/HRV_Personalized_sum